# Run MoneyPrinterTurbo on Google Colab

This notebook installs [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) in an isolated Python 3.11 environment and launches its WebUI. Colab runtimes are temporary, so generated files are removed when the runtime is reset.

## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/harry0703/MoneyPrinterTurbo.git"

# Update an existing checkout so rerunning this cell does not fail during clone.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")

## 2. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.

In [2]:
from getpass import getpass

from pyngrok import ngrok

# Close tunnels from earlier runs before configuring a new one.
ngrok.kill()

ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

ModuleNotFoundError: No module named 'pyngrok'

## 3. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys.

In [3]:
import time
from urllib.error import URLError
from urllib.request import urlopen

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
if "streamlit_proc" in globals() and streamlit_proc.poll() is None:
    streamlit_proc.terminate()
    streamlit_proc.wait(timeout=10)
if "streamlit_log" in globals() and not streamlit_log.closed:
    streamlit_log.close()
ngrok.kill()

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}", proto="http", bind_tls=True)
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")

NameError: name 'Path' is not defined